> 对应 Lec 13-15。覆盖 Ray Actor 工作节点设计模板、分布式 Lasso ADMM 完整代码（主节点循环 + 工作节点 Actor）、带全局正则项的通用框架、Cholesky 预分解优化、以及结果验证。算法推导见"8. 一致性优化_概念解释.md"；Ray Actor 基础见"2. Ray框架_实现.md"。

---

## Ray Actor 工作节点设计模板

### 算法思路

一致性 ADMM 把"$N$ 个节点共享同一个参数 $\theta$"的分布式问题改写成"每个节点有自己的局部副本 $\theta_i$，通过约束 $\theta_i=z$ 与全局共识 $z$ 对齐"的形式。每个工作节点在 $K$ 轮迭代中需要**持续保存**自己的数据块、局部参数、对偶变量，以及可复用的矩阵分解——这些状态如果用无状态的 Ray Task 实现，就必须每轮重新从头节点传输，浪费通信和计算。**Ray Actor**（`@ray.remote class`）正是为这种"长期存在、带内部状态的远程对象"设计的。

### __init__ 中保存的内容（数据、ρ、x_i、u_i、预计算量）

In [ ]:
import numpy as np
import ray
from scipy.linalg import cho_factor, cho_solve

@ray.remote
class ConsensusWorker:
    """
    一致性 ADMM 的工作节点 Actor。
    每个 Actor 对应一块本地数据 (A_i, b_i)，在多轮迭代中持久维护自己的状态。
    """
    def __init__(self, A_i, b_i, rho):
        self.A_i = A_i
        self.b_i = b_i
        self.rho = rho
        p = A_i.shape[1]

        self.x_i = np.zeros(p)   # 局部参数（本地子问题的解）
        self.u_i = np.zeros(p)   # 对偶变量（记录历史"一致性债务"）

        # 预计算 M = A_i^T A_i + rho*I 的 Cholesky 分解（只做一次！）
        self.M_cho = cho_factor(A_i.T @ A_i + rho * np.eye(p))
        # 预计算 q = A_i^T b_i（只做一次！）
        self.q = A_i.T @ b_i

**为什么这两项要在 `__init__` 里预计算？** 它们在整个迭代过程中都不会变化（$A_i, b_i, \rho$ 固定），如果放进每轮调用的方法里重复计算，等于每轮都浪费一次 $O(p^3)$ 的 Cholesky 分解——而真正必要的只是循环内 $O(p^2)$ 的回代。这是"避免循环内重复计算"原则在分布式场景下的体现。

### 预计算 M = A_i^TA_i + ρI 的 Cholesky 分解（只做一次）

### 预计算 q = A_i^Tb_i（只做一次）

见上方 `__init__` 代码。

---

## x_update 方法（工作节点第一步）

### 算法思路

每个工作节点独立求解一个**岭回归形式**的子问题：

$$x_i^{k+1} = \arg\min_{x_i}\left\{\frac12\|A_ix_i-b_i\|^2+\frac{\rho}{2}\|x_i-z^k+u_i^k\|^2\right\} = (A_i^TA_i+\rho I)^{-1}\left(A_i^Tb_i+\rho(z^k-u_i^k)\right)$$

直觉：本地损失项希望 $x_i$ 拟合本地数据，一致性惩罚项希望 $x_i$ 不要偏离全局共识 $z$ 太远（锚点是 $z^k-u_i^k$，即"全局共识减去历史偏差"）。这一步在所有节点之间**完全并行**，互不依赖。

### 构造右端向量：rhs = q + ρ(z - u_i)

### Cholesky 回代求解：cho_solve(M_cho, rhs)（O(p²)，比 solve 更快）

### 返回 (x_i, u_i)（主节点需要这两个量来计算 z）

In [ ]:
@ray.remote
class ConsensusWorker:
    # ...__init__ 同上...

    def x_update(self, z):
        """
        第一步：解本地子问题。
        只接收全局参数 z（p 维向量），不需要重传数据。
        """
        rhs = self.q + self.rho * (z - self.u_i)
        self.x_i = cho_solve(self.M_cho, rhs)   # O(p^2) 回代，远快于重新分解的 O(p^3)
        return self.x_i, self.u_i   # 主节点的 z-update 需要 x_i + u_i 的均值

**为什么要返回 `u_i`？** 主节点计算 $z^{k+1}$ 时需要 $\bar{x}^{k+1}+\bar{u}^k$（见下文 z-update），所以工作节点在这一步顺带把当前的 $u_i$（尚未更新）一起返回，省去一次额外的 RPC 调用。

---

## u_update 方法（工作节点第三步）

### 算法思路

对偶变量更新是一次纯粹的本地向量加法，记录"局部参数与全局共识之间历史累积的偏差"：

### 对偶变量更新：u_i ← u_i + x_i - z

### 返回本地原始残差²：‖x_i - z‖²（标量，用于收敛判断）

In [ ]:
@ray.remote
class ConsensusWorker:
    # ...

    def u_update(self, z_new):
        """
        第三步：更新对偶变量（本地操作，无需额外通信），
        并返回本地原始残差的平方，供主节点汇总判断收敛。
        """
        self.u_i = self.u_i + self.x_i - z_new
        return np.linalg.norm(self.x_i - z_new) ** 2   # 返回标量，而不是整个向量，减少通信量

**为什么返回平方而不是开根号后的范数？** 因为全局原始残差是 $\|r^k\|_2=\sqrt{\sum_i\|x_i-z\|^2}$——多个节点的平方项要先在主节点求和，再统一开一次根号。如果每个节点各自开根号再传回，主节点没法直接相加得到正确的全局范数。

---

## 分布式 Lasso ADMM 完整实现（Ray Actor）

### 算法思路

把 Lasso 目标 $\min_x\frac12\|Ax-b\|^2+\lambda\|x\|_1$ 按行分块到 $N$ 个节点：$f_i(x_i)=\frac12\|A_ix_i-b_i\|^2$（本地平方损失，可导），$g(z)=\lambda\|z\|_1$（全局正则项，放在 $z$ 上）。三步迭代与上面的通用 Actor 完全一致，只是 $z$-update 多了一步软阈值（因为 $g(z)\neq0$）。

### 创建 N 个工作节点 Actor（数据传输只在初始化时发生）

In [ ]:
def soft_thresholding(v, kappa):
    return np.sign(v) * np.maximum(np.abs(v) - kappa, 0.0)

def distributed_lasso_admm(A_blocks, b_blocks, lam, rho=1.0,
                            max_iter=200, eps=1e-4, verbose=0):
    """
    分布式 Lasso ADMM 主节点循环。

    Args:
        A_blocks, b_blocks: 长度为 N 的列表，每个元素是一个节点的本地数据块。
        lam: L1 正则化系数。
        rho: 增广拉格朗日系数。
    """
    N = len(A_blocks)
    p = A_blocks[0].shape[1]

    # 创建 N 个 Actor：数据只在这里传输一次，之后常驻在各自的远程节点上
    workers = [ConsensusWorker.remote(A_blocks[i], b_blocks[i], rho) for i in range(N)]

    z = np.zeros(p)
    z_prev = z.copy()

    for k in range(max_iter):
        # 第一轮广播：把当前 z^k 发给所有节点，并行调用 x_update
        xu_results = ray.get([w.x_update.remote(z) for w in workers])
        x_list = [r[0] for r in xu_results]
        u_list = [r[1] for r in xu_results]   # 这里取的是更新前的 u_i^k

        # z-update：x_bar = mean(x_list)，u_bar = mean(u_list)
        x_bar = np.mean(x_list, axis=0)
        u_bar = np.mean(u_list, axis=0)
        z_prev = z.copy()
        # 软阈值，阈值是 lam/(N*rho)：N 个约束合并后惩罚系数变成 N*rho
        z = soft_thresholding(x_bar + u_bar, lam / (N * rho))

        # 第二轮广播：把新的 z^{k+1} 发给所有节点，并行调用 u_update
        res_list = ray.get([w.u_update.remote(z) for w in workers])

        # 收敛判断：primal_res = sqrt(sum(残差^2))，dual_res = rho*sqrt(N)*||z变化量||
        primal_res = np.sqrt(sum(res_list))
        dual_res = rho * np.sqrt(N) * np.linalg.norm(z - z_prev)

        if verbose and k % 20 == 0:
            print(f"iter {k}: primal={primal_res:.6f}, dual={dual_res:.6f}")

        if primal_res < eps and dual_res < eps:
            print(f"第 {k+1} 轮收敛")
            break

    return k + 1, z

### 主节点迭代循环结构

### 第一轮广播 z^k 并行调用 x_update，收集 (x_i, u_i)

### z-update：x̄ = mean(x_list)，ū = mean(u_list)，z = S_{λ/(Nρ)}(x̄ + ū)

**为什么阈值是 $\lambda/(N\rho)$ 而不是单机版的 $\lambda/\rho$？** 单机 Lasso ADMM 只有一个约束 $\beta=z$，惩罚系数是 $\rho$；分布式版本有 $N$ 个约束 $x_i=z$（$i=1,\dots,N$）同时作用在 $z$ 上，$N$ 个 $\rho/2$ 惩罚项合并后系数变成 $N\rho/2$，相应地软阈值的阈值从 $\lambda/\rho$ 缩小为 $\lambda/(N\rho)$。这是"方差分解"推导（见"8. 一致性优化_概念解释.md"）的直接结果，节点数越多阈值越小，但最终的稀疏程度仍然由 $\lambda$ 本身决定。

### 第二轮广播 z^{k+1} 并行调用 u_update，收集局部残差²

### 收敛判断：primal_res = sqrt(Σ残差²)，dual_res = ρ·√N·‖z - z_prev‖

见上方完整代码。注意 `dual_res` 的计算完全在主节点本地完成（只需要 `z` 和 `z_prev`），不需要任何额外通信——这是一致性 ADMM 的一个工程优势：对偶残差不依赖工作节点的任何返回值。

---

## 带全局正则项的一致性 ADMM 通用框架

### 算法思路

把上面 Lasso 的具体实现抽象成通用框架：只要本地子问题 $f_i$ 可以用某种方式求解（不一定是闭式解，可以是迭代法），全局正则项 $g$ 替换成对应的近端算子，整个三步迭代结构保持不变。

### z-update 改为近端算子：prox_{g/(Nρ)}(x̄ + ū)

In [ ]:
def consensus_admm_general(workers, prox_g, p, rho=1.0, N=None,
                            max_iter=200, eps=1e-4):
    """
    通用带正则项一致性 ADMM 主循环。

    Args:
        workers: 已创建好的 Ray Actor 列表，每个都实现 x_update(z) 和 u_update(z_new) 方法。
        prox_g: 函数 prox_g(v, scale) -> 近端算子的结果。
                例如 Lasso 用 lambda v, scale: soft_thresholding(v, lam*scale)。
        N: 节点数。
    """
    N = N or len(workers)
    z = np.zeros(p)
    z_prev = z.copy()

    for k in range(max_iter):
        xu_results = ray.get([w.x_update.remote(z) for w in workers])
        x_bar = np.mean([r[0] for r in xu_results], axis=0)
        u_bar = np.mean([r[1] for r in xu_results], axis=0)

        z_prev = z.copy()
        # 唯一的框架差异：z-update 用 prox_g 代替具体的软阈值实现
        z = prox_g(x_bar + u_bar, 1.0 / (N * rho))

        res_list = ray.get([w.u_update.remote(z) for w in workers])
        primal_res = np.sqrt(sum(res_list))
        dual_res = rho * np.sqrt(N) * np.linalg.norm(z - z_prev)

        if primal_res < eps and dual_res < eps:
            break

    return k + 1, z

### 与无正则项版本（z = 均值）的唯一区别

Lec 13（无全局正则项，$g(z)=0$）的 $z$-update 就是简单的均值 $z^{k+1}=\bar{x}^{k+1}+\bar{u}^k$；这恰好是 $\text{prox}_{0}(v)=v$（零函数的近端算子就是恒等映射）。**两者在代码结构上完全一样**，唯一区别是 $z$-update 那一行调用的近端算子不同——这也是为什么上面的通用框架只需要传入不同的 `prox_g` 就能复用同一套主循环。

### Lasso 场景代入：g = λ‖·‖₁，prox = S_{λ/(Nρ)}

In [ ]:
# 调用方式：传入 Lasso 对应的近端算子（软阈值）
def make_lasso_prox(lam):
    return lambda v, scale: soft_thresholding(v, lam * scale)

iters, z_hat = consensus_admm_general(
    workers, prox_g=make_lasso_prox(lam=0.1), p=p, rho=1.0, N=N
)

---

## Cholesky 预分解优化（降低每轮迭代计算量）

### 算法思路

每个工作节点的 $x_i$-update 在数学上是反复求解同一个线性系统 $(A_i^TA_i+\rho I)x_i = \text{rhs}$，只是右端向量 `rhs` 每轮都不同。利用这一点，把"分解系数矩阵"和"求解线性系统"拆成两步：分解只做一次（系数矩阵不变），每轮只做开销小得多的回代。

### 初始化时：from scipy.linalg import cho_factor; M_cho = cho_factor(A.T@A + ρ*I)

### 每轮迭代：from scipy.linalg import cho_solve; x = cho_solve(M_cho, rhs)

In [ ]:
from scipy.linalg import cho_factor, cho_solve

# 初始化阶段（只执行一次，O(p^3)）
M_cho = cho_factor(A_i.T @ A_i + rho * np.eye(p))

# 每轮迭代（执行 max_iter 次，每次只需 O(p^2)）
for k in range(max_iter):
    rhs = q + rho * (z - u_i)
    x_i = cho_solve(M_cho, rhs)
    # ...后续 u_i 更新等...

### 复杂度：初始化 O(p³) 一次，每轮 O(p²)（vs 不预分解每轮 O(p³)）

如果不做预分解，每轮都用 `np.linalg.solve(A_i.T @ A_i + rho * np.eye(p), rhs)`，等价于每轮都重新做一次 $O(p^3)$ 的分解——当 `max_iter` 较大（几百到上万轮）时，这个差异会被放大成数量级的速度差距。

### ρ 改变时必须重做 Cholesky（M = A^TA + ρI 随 ρ 变化）

**易错点**：Cholesky 分解缓存的前提是 $M=A_i^TA_i+\rho I$ 在迭代过程中**不变**。如果算法设计中 $\rho$ 会自适应调整（比如某些自适应 ADMM 变体会根据残差比例动态调节 $\rho$），那么每次 $\rho$ 改变后都必须重新调用 `cho_factor`，否则用旧的分解结果求解会得到错误的 $x_i$。本课程范围内的 ADMM 默认 $\rho$ 固定，不需要处理这种情况，但理解这个边界条件本身是常考的概念题。

---

## 分布式 Lasso 结果验证

### 算法思路

写完分布式实现后，需要用几种方式确认结果正确，而不是仅凭"代码跑完没报错"就认为对了。

### 与 sklearn Lasso（alpha=λ/n）结果对比

**为什么是 `alpha=lam/n` 而不是 `alpha=lam`？** sklearn 的 `Lasso` 内部对目标函数做了 $1/n$ 归一化：

$$\text{sklearn 目标} = \frac{1}{2n}\|y-X\beta\|^2+\alpha\|\beta\|_1$$

而本课程的推导用的是未归一化形式 $\frac12\|y-X\beta\|^2+\lambda\|\beta\|_1$。要让两者等价，需要令 $\alpha = \lambda/n$。

In [ ]:
from sklearn.linear_model import Lasso

n_total = sum(A_i.shape[0] for A_i in A_blocks)
sk_model = Lasso(alpha=lam / n_total, fit_intercept=False)
sk_model.fit(np.vstack(A_blocks), np.concatenate(b_blocks))

print("分布式 ADMM:", z_hat[:5])
print("sklearn:    ", sk_model.coef_[:5])
print("最大绝对误差:", np.max(np.abs(z_hat - sk_model.coef_)))

### 非零系数数量验证（np.sum(np.abs(beta) > 1e-6)）

In [ ]:
n_nonzero = np.sum(np.abs(z_hat) > 1e-6)
print(f"非零系数个数: {n_nonzero} / {len(z_hat)}")
# 如果生成数据时已知真实 beta 的稀疏模式（如后10个系数为0），
# 可以直接比较恢复的稀疏模式是否一致

### 估计误差（‖β̂ - β_true‖）随迭代轮次的收敛曲线

In [ ]:
import matplotlib.pyplot as plt

# 在主循环每轮记录一次误差（需要在 distributed_lasso_admm 内部加一行记录逻辑）
errors = []
# ... 在循环内: errors.append(np.linalg.norm(z - beta_true)) ...

plt.plot(errors)
plt.xlabel("迭代轮次")
plt.ylabel("||beta_hat - beta_true||")
plt.yscale("log")   # 残差通常跨越多个数量级，对数坐标更容易观察收敛趋势
plt.title("分布式 Lasso ADMM 收敛曲线")
plt.show()

**验证三件套总结**：(1) 与成熟库的结果数值对比，确认实现没有系统性错误；(2) 检查稀疏模式是否符合预期，确认 L1 正则确实起作用；(3) 画收敛曲线，确认算法确实在收敛而不是震荡或发散。三者缺一不可，只看最终系数容易掩盖中间过程的问题（如 $\rho$ 设置不当导致收敛极慢但凑巧在 `max_iter` 截断处接近真值）。